implement strategy_pattern.ipynb for cross validation in a sheet/table (conditional validation)

In [ ]:
from abc import ABC, abstractmethod
from typing import Dict, List, Union, Optional, Any
from loguru import logger
import pandas as pd
import numpy as np

class ValidationStrategy(ABC):

    def __init__(self, config: Optional[Dict] = None):
        self.config = config or {}

    @abstractmethod
    def validate(
        self,
        df: pd.DataFrame,
        column: str,
        *args,
        **kwargs
        ) -> pd.Series:
        '''Return vectorized validation result. True for invalid, False for valid.'''
        pass


In [ ]:
def is_empty(data: pd.Series) -> pd.Series:
    """Return True if df is empty or None else False."""
    return data is None or data.empty

In [ ]:
class MandatoryValidation(ValidationStrategy):

    def __init__(self, config: Optional[Dict] = None, *args, **kwargs):
        super().__init__(config)

    def validate(self, df: pd.DataFrame, column: str, empty_list: Optional[List] = None) -> pd.Series:
        mask: pd.Series = df[column].isna()
        if empty_list:
            mask = mask | (df[column].isin(empty_list))
        return mask

In [ ]:
import yaml
import json
import pandas as pd

with open("config.yml", "r") as f:
    data = yaml.load(f, Loader=yaml.FullLoader)
    # df = pd.DataFrame(data)
data

In [ ]:
internal_suspended_list_validation_config = data["files"][0]
internal_suspended_list_validation_config

In [ ]:
internal_suspended_list_rule_config = pd.DataFrame(internal_suspended_list_validation_config["columns"])
internal_suspended_list_rule_config

In [ ]:
required_columns = internal_suspended_list_rule_config.loc[internal_suspended_list_rule_config["required"], "name"]
required_columns

In [ ]:
master_data_setup_file_path = "/home/user/data-da-ds-de/data_validation_pandas/data_test/raw/prevalidation_data_2503/Data Template 03_Master Data Setup_SLP3P4 1.xlsx"
df_internal_suspended_list: pd.DataFrame = pd.read_excel(master_data_setup_file_path, sheet_name="InternalSuspendedList")
df_internal_suspended_list["validation_result"] = [set() for i in range(df_internal_suspended_list.shape[0])]

In [ ]:
if is_empty(df_internal_suspended_list):
    logger.error("The InternalSuspendedList sheet is empty.")
    # End program

In [ ]:
# Using Factory pattern to registry validation strategy

class ValidationStrategyFactory:
    def __init__(self, context: Optional[Dict] = None):
        self.strategies: Dict[str, ValidationStrategy] = {}
        self.context = context

    def register(self, name: str, strategy: ValidationStrategy):
        self.strategies[name] = strategy

    def get_strategy(self, name: str) -> ValidationStrategy:
        return self.strategies[name]

    def build_strategy(self, name: str, config: Optional[Dict] = None) -> ValidationStrategy:
        rule_type = config.get("type")
        rule_class = self.strategies.get(rule_type)
        logger.info(list(self.strategies.keys()))
        if rule_type not in list(self.strategies.keys()):
            raise ValueError(f"Unknown rule type: {rule_type}.")

        if rule_class is None:
            raise ValueError(f"Rule class not found for rule type: {rule_type}.")
        if self.context is None:
            self.context = {}
        return rule_class(config, **self.context)

# Using Factory pattern to registry validation strategy
validation_strategy_factory = ValidationStrategyFactory()

mandatory_validation_strategy = validation_strategy_factory.register("mandatory", MandatoryValidation)
mandatory_validation_strategy = validation_strategy_factory.get_strategy("mandatory")

In [ ]:
df_internal_suspended_list["validation_result"] = [set() for i in range(df_internal_suspended_list.shape[0])]
empty_value_list = ["", "nan", np.nan, pd.NA, None]
for column in required_columns:
    # Pre procressing
    df_internal_suspended_list[column] = df_internal_suspended_list[column].str.strip()
    # Validation
    validation_result = mandatory_validation_strategy().validate(df=df_internal_suspended_list, column=column, empty_list=empty_value_list)
    # Record result
    df_internal_suspended_list.loc[validation_result, "validation_result"] = df_internal_suspended_list.loc[validation_result, "validation_result"].map(lambda x: x.union({f"{column} is mandatory"}))
    # Log result
    logger.info(f"[{column}] [Check mandatory] {validation_result.sum()}/{df_internal_suspended_list.shape[0]} records invalid, Excel index example: {df_internal_suspended_list.loc[validation_result, column].index[:5].tolist()}")
# df_internal_suspended_list

In [ ]:
# for i in internal_suspended_list_validation_config:
# validation follow config

In [ ]:
nric_fin_validation_config =internal_suspended_list_validation_config["columns"][2]
nric_fin_validation_config
nric_fin_validation_rules = nric_fin_validation_config["rules"]
import json
print(json.dumps(nric_fin_validation_rules, indent=2))


In [ ]:
import sys
import snoop
snoop.install(out=sys.stdout, overwrite=True)

In [ ]:
import operator
import snoop

class ConditionParser:

    OPERATOR_MAP = {
        "==": operator.eq,
        "!=": operator.ne,
        ">": operator.gt,
        "<": operator.lt,
        ">=": operator.ge,
        "<=": operator.le,
        "in": lambda x, y: x.isin(y),
        "not in": lambda x, y: ~x.isin(y),
    }

    LOGICAL_MAP = {
        "AND": lambda a, b: a & b,
        "OR": lambda a, b: a | b,
        "XOR": lambda a, b: a ^ b,
        "NOT": lambda a: ~a,
        "NOR": lambda a, b: ~(a | b),
        "NAND": lambda a, b: ~(a & b),
        "XNOR": lambda a, b: ~(a ^ b),
    }

    @classmethod
    @snoop
    def build_mask(cls, df: pd.DataFrame, condition_config: dict) -> pd.Series:
        """
        Ex:
        {
          "operator": "AND",
          "conditions": [
            {
              "column": "Debtor type",
              "operator": "==",
              "values": "Individual"
            },
            {
              "column": "Status",
              "operator": "==",
              "values": "Active"
            }
          ],
        """

        if "conditions" in condition_config:
            logger.info(condition_config)
            # Logical group
            operator_key = condition_config["operator"]
            conditions = condition_config["conditions"]

            masks = [cls.build_mask(df, c) for c in conditions] # build mask, meaning check each condition, return True or False each cell, mask[0] is the first condition, mask[1] is the second condition or result of group condition,... It mask from down to up. For example, mask[0] return True/False of Debtor type == Individual, mask[1] return True/False of Status == Active, then apply operator AND to get final result. If it has deeper, ex: (Debtor type == Individual) AND ((Status == Active) OR (Debtor type == Company)), mask[0] is top condition, mask[1] is result of ((Status == Active) OR (Debtor type == Company)). Final result is mask[0] & mask[1].

            result = masks[0]
            for mask in masks[1:]:
                result = cls.LOGICAL_MAP[operator_key](result, mask)

            return result

        else:
            # Leaf condition
            logger.info(condition_config)
            logger.info(df.columns)
            column = condition_config["column"]
            op = condition_config["operator"]
            values = condition_config["values"]

            if op not in cls.OPERATOR_MAP:
                raise ValueError(f"Unsupported operator: {op}")

            return cls.OPERATOR_MAP[op](df[column], values)

In [ ]:
config = {
          "operator": "AND",
          "conditions": [
            {
              "column": "Debtor type",
              "operator": "==",
              "values": "Individual"
            },
            {
              "column": "Status",
              "operator": "==",
              "values": "Active"
            }
          ],
        }
mask = ConditionParser.build_mask(df_internal_suspended_list, config)
df_internal_suspended_list[mask]

In [ ]:
# operator.eq(df_internal_suspended_list["ID"], "123")

In [ ]:
class InnerReferenceValidation(ValidationStrategy):
    def __init__(
      self,
      config: Optional[Dict] = None,
      factory: Optional[ValidationStrategyFactory] = None
      ) -> None:
      self.ref_info = config["ref_info"]
      self.factory = factory
      self.compiled_refs = []

      for ref in self.ref_info:
          condition = ref.get("condition")
          sub_rules = [
                self.factory.build_strategy(rule.get("type"), rule)
                for rule in ref.get("ref_rules", [])
            ]
          self.compiled_refs.append({
                "condition": condition,
                "rules": sub_rules
            }) # recursion

    def validate(
      self,
      df: pd.DataFrame,
      column: str,
      *args,
      **kwargs
      ) -> pd.Series:
        final_mask = pd.Series(False, index=df.index)

        for ref in self.compiled_refs:
            logger.info(f"Running inner reference: {ref}")
            condition_mask = ConditionParser.build_mask(df, ref["condition"])

            for rule in ref["rules"]:
                sub_mask = rule.validate(df, column)
                # Apply only where condition is true
                invalid_mask = condition_mask & sub_mask
                final_mask = final_mask | invalid_mask

        return final_mask


In [ ]:
validation_strategy_factory.strategies

In [ ]:
nric_fin_validation_config["rules"][1].keys()

In [ ]:
validation_strategy_factory.register("inner_reference", InnerReferenceValidation)
inner_validation_strategy = validation_strategy_factory.get_strategy("inner_reference")

inner_validation_strategy(config=nric_fin_validation_config["rules"][1], factory=validation_strategy_factory).validate(df_internal_suspended_list, "ID")

In [ ]:
df_internal_suspended_list.loc[(df_internal_suspended_list["Debtor type"] == "Individual") & (df_internal_suspended_list["Status"] == "Active"), "ID"].isna().value_counts()

## outer reference validation

In [ ]:
class FileLoaderStrategy(ABC):
    def __init__(
        self,
        file_path: str,
        *args,
        **kwargs
        ) -> None:
        self.file_path = file_path
        self.args = args or ()
        self.kwargs = kwargs or {}

    @abstractmethod
    def load(
        self,
        *args,
        **kwargs
        ) -> pd.DataFrame | pd.Series:
        pass

class ExcelLoaderStrategy(FileLoaderStrategy):
    def __init__(
        self,
        file_path: str,
        sheet_name: str,
        *args,
        **kwargs
        ) -> None:
        super().__init__(file_path, *args, **kwargs)
        self.file_path = file_path
        self.sheet_name = sheet_name

        if not self.file_path:
            raise ValueError("File path is required")
        if not self.sheet_name:
            raise ValueError("Sheet name is required")

    def load(
        self,
        *args,
        **kwargs
        ) -> pd.DataFrame:
        sheet_names = pd.ExcelFile(self.file_path).sheet_names
        if self.sheet_name not in sheet_names:
            raise ValueError(f"Sheet name {self.sheet_name} not found in file {self.file_path}")

        self.args = (*self.args, *args)
        self.kwargs = {**self.kwargs, **kwargs}

        return pd.read_excel(
            self.file_path,
            sheet_name=self.sheet_name,
            *self.args,
            **self.kwargs
        )

class CSVLoaderStrategy(FileLoaderStrategy):
    def __init__(
        self,
        file_path: str,
        *args,
        **kwargs
        ) -> None:
        super().__init__(file_path, *args, **kwargs)

    def load(
        self,
        *args,
        **kwargs
        ) -> pd.DataFrame:
        return pd.read_csv(
            self.file_path,
            *args,
            **kwargs
    )


class ParquetLoaderStrategy(FileLoaderStrategy):
    def __init__(
        self,
        file_path: str,
        *args,
        **kwargs
        ) -> None:
        super().__init__(file_path, *args, **kwargs)

    def load(
        self,
        *args,
        **kwargs
        ) -> pd.DataFrame:
        return pd.read_parquet(
            self.file_path,
            *args,
            **kwargs
    )

In [ ]:
class FileLoaderStrategyFactory:
    def __init__(self) -> None:
        self.strategies = {}

    def register(self, name: str, strategy: FileLoaderStrategy):
        self.strategies[name] = strategy

    def get_strategy(self, name: str) -> FileLoaderStrategy:
        if name not in self.strategies:
            raise ValueError(f"Strategy {name} not found")
        return self.strategies[name]

    def build_strategy(self, name: str, *args, **kwargs) -> FileLoaderStrategy:
        return self.strategies[name](*args, **kwargs)


file_loader_strategy_factory = FileLoaderStrategyFactory()
file_loader_strategy_factory.register("excel", ExcelLoaderStrategy)
file_loader_strategy_factory.register("csv", CSVLoaderStrategy)
file_loader_strategy_factory.register("parquet", ParquetLoaderStrategy)


In [ ]:
excel_file_loader_strategy = file_loader_strategy_factory.get_strategy("excel")
excel_file_loader_strategy(file_path=master_data_setup_file_path, sheet_name="InternalSuspendedList", usecols=["ID"]).load()

In [ ]:
parquet_file_loader_strategy = file_loader_strategy_factory.get_strategy("parquet")
parquet_file_loader_strategy(file_path="/home/user/data-da-ds-de/data_validation_pandas/data_test/raw/parquet/2026-02-06/Data Template 03_Master Data Setup_SLP3P4 1/SuspensionReason.parquet").load()

In [ ]:
# file_path: str = ""
# if file_path.endswith(".xlsx") or file_path.endswith(".xls"):
#     file_loader_strategy_factory.build_strategy("excel", file_path=file_path)
# elif file_path.endswith(".csv"):
#     file_loader_strategy_factory.build_strategy("csv", file_path=file_path)
# elif file_path.endswith(".parquet"):
#     file_loader_strategy_factory.build_strategy("parquet", file_path=file_path)
# else:
#     raise ValueError("File type not supported")

In [ ]:
class OuterReferenceRegistry:
    def __init__(self, file_loader_strategy: FileLoaderStrategy) -> None:
        self.file_loader_strategy = file_loader_strategy
        self.cache = {}

    def get_column_values(
        self,
        file_name: str,
        column: str,
        *args,
        **kwargs
        ) -> pd.Series:

        key = (file_name, column)

        if key not in self.cache:
            df = self.file_loader(file_name=file_name, *args, **kwargs)
            self.cache[key] = df[column].dropna().drop_duplicates()

        return self.cache[key]



In [ ]:
import yaml
import json
import pandas as pd

with open("config.yml", "r") as f:
    data = yaml.load(f, Loader=yaml.FullLoader)
susrension_reason_name_outer_ref_conig = data["files"][0]["columns"][3]["rules"][1]
print(json.dumps(susrension_reason_name_outer_ref_conig, indent=1))

In [ ]:
import snoop
snoop.install(color=True)
class OuterReferenceValidation(ValidationStrategy):
    def __init__(
      self,
      config: Optional[Dict] = None,
      factory: Optional[ValidationStrategyFactory] = None,
      # outer_reference_registry: Optional[OuterReferenceRegistry] = None
      ) -> None:
      self.ref_info = config["ref_info"]
      self.factory = factory
      # self.outer_reference_registry = outer_reference_registry
      self.compiled_refs = []

      if not self.ref_info:
        raise ValueError("ref_info is required")

      for ref in self.ref_info:
          condition = ref.get("condition")
          sub_rules = [
                self.factory.build_strategy(rule.get("type"), rule)
                for rule in ref.get("ref_rules", [])
            ]

          self.compiled_refs.append({
                "condition": condition,
                "rules": sub_rules
            }) # recursion

    @snoop
    def load_outer_reference_data(
        self,
        file_path: str,
        *args,
        **kwargs
        ) -> pd.DataFrame:
        file_type = file_path.split(".")[-1]
        file_loader_strategy_factory = FileLoaderStrategyFactory()
        file_loader_strategy_factory.register("excel", ExcelLoaderStrategy)
        file_loader_strategy_factory.register("csv", CSVLoaderStrategy)
        file_loader_strategy_factory.register("parquet", ParquetLoaderStrategy)

        if file_type == "csv":
            file_loader_strategy = file_loader_strategy_factory.get_strategy("csv", file_path=file_path)
            return file_loader_strategy(*args, **kwargs).load()
        if file_type == "parquet":
            file_loader_strategy = file_loader_strategy_factory.get_strategy("parquet", file_path=file_path)
            return file_loader_strategy(*args, **kwargs).load()
        if file_type == "xlsx" or file_type == "xls":
            file_loader_strategy = file_loader_strategy_factory.get_strategy("excel", file_path=file_path)
            return file_loader_strategy(*args, **kwargs).load()
        else:
            file_loader_strategy_factory.register(file_type, file_path=file_path)
            file_loader_strategy = file_loader_strategy_factory.get_strategy(file_type, file_path=file_path)
            return file_loader_strategy(*args, **kwargs).load()

    @snoop
    def reference_mask(self, df: pd.DataFrame, current_column: str, config: Dict) -> pd.Series:
        operator = config["operator"].upper()
        conditions = config["conditions"]

        masks = []

        for condition in conditions:
            file_path: str = condition["file_path"]
            sheet_name: str = condition["sheet_name"]
            ref_column = condition["column"]
            filters = condition["conditions"]

            df_ref = self.load_outer_reference_data(file_path=file_path, sheet_name=sheet_name)
            logger.info(df_ref)

            if filters:
                for f in filters:
                    mask_ref = ConditionParser.build_mask(df_ref, f)
                    df_ref = df_ref.loc[mask_ref]

            if ref_column not in df.columns:
                raise ValueError(f"Column {ref_column} not found in reference dataframe")

            lookup_set = df_ref[ref_column].dropna().drop_duplicates()

            mask = df[current_column].isin(lookup_set)

            masks.append(mask)

        if not masks:
          return pd.Series(False, index=df.index)

        result = masks[0]
        for mask in masks[1:]:
          result = ConditionParser.LOGICAL_MAP[operator](result, mask)

        return result

    @snoop
    def validate(
      self,
      df: pd.DataFrame,
      column: str,
      *args,
      **kwargs
      ) -> pd.Series:
        final_mask = pd.Series(False, index=df.index)

        for ref in self.compiled_refs:

            condition_mask = self.reference_mask(df, column, ref["condition"])

            # Apply nested rules only when condition true
            for rule in ref["rules"]:

                sub_mask = rule.validate(df)

                invalid_mask = condition_mask & sub_mask

                final_mask = final_mask | invalid_mask

        return final_mask

In [ ]:
validation_strategy_factory.register("outer_reference", InnerReferenceValidation)
outer_validation_strategy = validation_strategy_factory.get_strategy("outer_reference")

outer_validation_strategy(
    config=susrension_reason_name_outer_ref_conig,
    factory=validation_strategy_factory
    ).validate(df_internal_suspended_list, "ID")